In [7]:
import os
import mlflow
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, accuracy_score

def train_clause_classifier():
    # Set up local MLflow tracking infrastructure
    mlflow.set_tracking_uri("sqlite:///mlflow.db")
    mlflow.set_experiment("Enterprise_Compliance_Classifier")
    
    with mlflow.start_run() as run:
        # 1. Log Architecture & Hyperparameters
        params = {"C": 1.0, "max_iter": 200, "classifier_type": "LogisticRegression_Top_Layer"}
        mlflow.log_params(params)
        
        # 2. Simulate feature matrices from our Step 2 Embeddings (English & Arabic vectors)
        X_train = np.random.rand(100, 768)  # 100 sample documents, 768 LaBSE dimensions
        y_train = np.random.choice([0, 1], size=100) # 0: Low Risk, 1: High Risk Clause
        
        X_val = np.random.rand(20, 768)
        y_val = np.random.choice([0, 1], size=20)
        
        # 3. Model Fit
        model = LogisticRegression(C=params["C"], max_iter=params["max_iter"])
        model.fit(X_train, y_train)
        
        # 4. Evaluate Validation Split
        predictions = model.predict(X_val)
        acc = accuracy_score(y_val, predictions)
        f1 = f1_score(y_val, predictions, average='macro')
        
        # 5. Log Metrics to MLflow Dashboard
        mlflow.log_metric("validation_accuracy", acc)
        mlflow.log_metric("validation_f1_macro", f1)
        
        # 6. Save and Register the Model
        mlflow.sklearn.log_model(
            sk_model=model,
            artifact_path="compliance_model",
            registered_model_name="Bilingual_Clause_Risk_Classifier"
        )
        
        print(f"Run tracked successfully. Run ID: {run.info.run_id}")
        print(f"Metrics logged -> Accuracy: {acc}, F1: {f1}")

if __name__ == "__main__":
    train_clause_classifier()


2026/10/13 12:38:20 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/10/13 12:38:20 INFO mlflow.store.db.utils: Updating database tables
2026/10/13 12:38:21 INFO mlflow.tracking.fluent: Experiment with name 'Enterprise_Compliance_Classifier' does not exist. Creating a new experiment.
2026/10/13 12:38:22 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Run tracked successfully. Run ID: 1e4bfa5272664d3bb2af93e985dff731
Metrics logged -> Accuracy: 0.55, F1: 0.5396419437340154


Successfully registered model 'Bilingual_Clause_Risk_Classifier'.
Created version '1' of model 'Bilingual_Clause_Risk_Classifier'.


In [8]:
# mlflow server --backend-store-uri sqlite:///mlflow.db
# Run the above command in the same directory in cmd